In [6]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sympy import lambdify
import sympy as sp

# 1. Point Python to your exact SRC directory
sys.path.append("/Users/adarshmac/simulations/bbh_run1/ANALYTICAL_CODES/SRC")

# 2. Import the engine AND EVERY MASKED metric available in your new library
from MaskerMetricstest import (
    calculate_automated_fields,
    get_schwarzschild_spherical_masked,
    get_schwarzschild_isotropic_cartesian_masked,
    get_schwarzschild_pg_cartesian_masked,
    get_interior_schwarzschild_spherical_masked,
    get_kerr_boyer_lindquist_masked,
    get_kerr_schild_cartesian_masked,
    get_reissner_nordstrom_spherical_masked,
    get_bardeen_cartesian_masked,
    get_flrw_cartesian_masked
)

# --- The Unmasking Helper Function ---
def unmask_tensor(tensor, subs_dict):
    """Safely plugs the real math back into the dummy tensor"""
    if not subs_dict: return tensor
    if hasattr(tensor, 'applyfunc'):
        return tensor.applyfunc(lambda expr: expr.subs(subs_dict).doit() if expr != 0 else sp.sympify(0))
    return [expr.subs(subs_dict).doit() if expr != 0 else sp.sympify(0) for expr in tensor]
# ------------------------------------------

# 3. Load the MASKED geometry (Change this line to run different metrics)
print("Loading Metric Geometry...")
metric_data = get_kerr_boyer_lindquist_masked()  

# 4. Execute the engine (Will run instantly because of the mask)
print("Executing Tensor Calculus Engine...")
results = calculate_automated_fields(metric_data)

# 5. Extract AND UNMASK all matrices 
print("Unmasking tensors...")
subs = metric_data.get('subs_dict', {})

E_hat = unmask_tensor(results['E_hat'], subs)
D_hat = unmask_tensor(results['D_hat'], subs)
B_hat = unmask_tensor(results['B_hat'], subs)
H_hat = unmask_tensor(results['H_hat'], subs)
rho_hat = unmask_tensor(results['rho_hat'], subs)
q_hat = unmask_tensor(results['q_hat'], subs)
s_hat = unmask_tensor(results['s_hat'], subs)
j_hat = unmask_tensor(results['j_hat'], subs)

# 6. UNIVERSAL SYMBOL EXTRACTION
print("Extracting variables...")
syms = results['symbols']

# Safely extract all potential constants
M = syms.get('M')
a = syms.get('a')
Q = syms.get('Q')
Qm = syms.get('Qm')
R = syms.get('R')

# Safely extract all potential coordinates
x, y, z = syms.get('x'), syms.get('y'), syms.get('z')
r, theta, phi = syms.get('r'), syms.get('theta'), syms.get('phi')
r_bar = syms.get('r_bar')

print("Ready for printing!")

Loading Metric Geometry...
Executing Tensor Calculus Engine...
Unmasking tensors...
Extracting variables...
Ready for printing!


In [7]:
# ==========================================
# MODULE 9: ANALYTICAL EXPRESSION EXPORTER
# ==========================================
import sympy as sp

def export_latex_expressions(tensor, name_symbol, is_spatial_matrix=False, index_style="down"):
    print(f"\n% --- LaTeX Output for {name_symbol} ---")
    print(r"\begin{align}")
    
    non_zero_elements = []
    indices = []
    
    # 1. Collect non-zero elements WITHOUT using sp.cancel (prevents freezing)
    if hasattr(tensor, 'shape') and len(tensor.shape) == 2:
        rows, cols = tensor.shape
        row_start, col_start = (1, 1) if is_spatial_matrix else (0, 0)
        
        for i in range(row_start, rows):
            for j in range(col_start, cols):
                expr = tensor[i, j]
                if expr != 0:
                    non_zero_elements.append(expr)
                    if index_style == "up": idx_str = f"^{{{i}{j}}}"
                    elif index_style == "mixed": idx_str = f"^{{{i}}}_{{{j}}}"
                    else: idx_str = f"_{{{i}{j}}}"
                    indices.append(idx_str)
                    
    else: 
        for i in range(len(tensor)):
            expr = tensor[i]
            if expr != 0:
                non_zero_elements.append(expr)
                idx_str = f"^{{{i}}}" if index_style == "up" else f"_{{{i}}}"
                indices.append(idx_str)

    if not non_zero_elements:
        print(f"    % All components of {name_symbol} are zero.")
        print(r"\end{align}")
        print("% ----------------------------------------\n")
        return

    # 2. Use CSE to clean up the math safely
    replacements, reduced_exprs = sp.cse(non_zero_elements)
    
    if replacements:
        print("    % --- Intermediate Variables (CSE) --- \\\\")
        for var, sub_expr in replacements:
            print(f"    {sp.latex(var)} &= {sp.latex(sub_expr)} \\\\")
        print("    % ------------------------------------ \\\\")
        
    for idx_str, red_expr in zip(indices, reduced_exprs):
        print(f"    {name_symbol}{idx_str} &= {sp.latex(red_expr)} \\\\")
        
    print(r"\end{align}")
    print("% ----------------------------------------\n")

# 7. EXPORT TO LATEX
print("Generating Optimized LaTeX...")
export_latex_expressions(rho_hat, r"\rho", index_style="down")
export_latex_expressions(q_hat, "q", index_style="down")
export_latex_expressions(E_hat, "E", is_spatial_matrix=False, index_style="up")
export_latex_expressions(B_hat, "B", is_spatial_matrix=True, index_style="up")
export_latex_expressions(D_hat, "D", is_spatial_matrix=False, index_style="mixed")
export_latex_expressions(H_hat, "H", is_spatial_matrix=True, index_style="mixed")
export_latex_expressions(s_hat, "s", is_spatial_matrix=True, index_style="mixed")
export_latex_expressions(j_hat, "j", is_spatial_matrix=True, index_style="mixed")

Generating Optimized LaTeX...

% --- LaTeX Output for \rho ---
\begin{align}
    % --- Intermediate Variables (CSE) --- \\
    x_{0} &= a^{2} \\
    x_{1} &= r^{2} \\
    x_{2} &= x_{0} + x_{1} \\
    x_{3} &= 2 M \\
    x_{4} &= r x_{3} \\
    x_{5} &= x_{2} - x_{4} \\
    x_{6} &= \sin{\left(\theta \right)} \\
    x_{7} &= x_{6}^{2} \\
    x_{8} &= x_{0} x_{7} \\
    x_{9} &= x_{2}^{2} - x_{5} x_{8} \\
    x_{10} &= \frac{1}{x_{9}} \\
    x_{11} &= \cos{\left(\theta \right)} \\
    x_{12} &= x_{0} x_{11}^{2} + x_{1} \\
    x_{13} &= \frac{1}{x_{12}} \\
    x_{14} &= x_{13} x_{9} \\
    x_{15} &= \sqrt{x_{14}} \\
    x_{16} &= \frac{1}{x_{15}} \\
    x_{17} &= 4 r \\
    x_{18} &= 2 r - x_{3} \\
    x_{19} &= - x_{17} x_{2} + x_{18} x_{8} \\
    x_{20} &= x_{12}^{2} \\
    x_{21} &= \frac{1}{x_{20}} \\
    x_{22} &= x_{21} x_{9} \\
    x_{23} &= x_{16} \left(- 2 r x_{22} x_{7} - x_{13} x_{19} x_{7}\right) \\
    x_{24} &= \left|{x_{6}}\right| \\
    x_{25} &= \frac{M r}{x_{24}} \\
   